# Day 10: Visualizing Attention in Transformers (GPT-2)

Welcome to Day 10 of your AI Engineering journey! Today we are looking under the hood of Large Language Models to understand how they process context.

## Core Theory (Just-in-Time)

**What is Attention?**
In traditional Sequence-to-Sequence models (like older RNNs), processing long context was difficult because the model had to compress the entire past sequence into a single fixed-size hidden state. 

The **Transformer** architecture solved this with the *Self-Attention* mechanism. Attention allows the model to look at all words in the input sequence simultaneously and assign a "weight" or "importance" to each word when processing the current word. 

**How does it work?**
For each token, the model creates three vectors: Query (Q), Key (K), and Value (V). 
1.  The **Query** is what the current token is looking for.
2.  The **Key** is what other tokens offer.
3.  The dot product of Q and K (scaled and softmaxed) creates the **Attention Weights** (a probability distribution showing where the model is "looking").
4.  These weights are multiplied by the **Value** vectors to produce the final context-aware representation.

In GPT-2, this is *Causal Masked Self-Attention* — a token can only "attend" to itself and previous tokens, never future tokens (since it's an autoregressive model).

**Why Visualize It?**
Visualizing these weights helps us understand the model's reasoning. We can see if a pronoun is correctly attending to its noun (coreference resolution) or how far back the model looks for context.

## Code Implementation

Let's load a pre-trained GPT-2 model and extract its attention weights. We will use the officially supported `transformers` library.

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import GPT2Tokenizer, GPT2Model
from typing import Tuple, List, Dict, Any

def load_model_and_tokenizer(model_name: str = "gpt2") -> Tuple[GPT2Model, GPT2Tokenizer]:
    """
    Loads the pre-trained GPT-2 model and tokenizer.
    
    Args:
        model_name: The identifier of the model on Hugging Face hub.
        
    Returns:
        A tuple containing the loaded model and tokenizer.
    """
    print(f"Loading {model_name}...")
    # We must set output_attentions=True to extract the attention weights
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    model = GPT2Model.from_pretrained(model_name, output_attentions=True)
    model.eval() # Set to evaluation mode (disable dropout, etc.)
    return model, tokenizer

def get_attention_weights(
    text: str, 
    model: GPT2Model, 
    tokenizer: GPT2Tokenizer
) -> Tuple[torch.Tensor, List[str]]:
    """
    Processes text through the model and extracts attention weights.
    
    Args:
        text: The input string to process.
        model: The GPT-2 model.
        tokenizer: The corresponding tokenizer.
        
    Returns:
        A tuple containing the attention weights tensor and the list of string tokens.
        Attention tensor shape: (num_layers, batch_size, num_heads, sequence_length, sequence_length)
    """
    # Tokenize input and convert to PyTorch tensors
    inputs = tokenizer(text, return_tensors="pt")
    
    # Get the token strings for visualization
    # GPT-2 uses Byte-Pair Encoding, tokens might start with 'Ġ' indicating a space
    input_ids = inputs["input_ids"][0]
    tokens = [tokenizer.decode([token_id]).strip() for token_id in input_ids]
    
    # Forward pass without gradient calculation (saves memory/compute)
    with torch.no_grad():
        outputs = model(**inputs)
        
    # outputs.attentions is a tuple of tensors, one for each layer.
    # We stack them into a single tensor.
    attentions = torch.stack(outputs.attentions)
    
    return attentions, tokens

def plot_attention_head(
    attention: torch.Tensor, 
    tokens: List[str], 
    layer: int, 
    head: int
) -> None:
    """
    Plots the attention matrix for a specific layer and head.
    
    Args:
        attention: The stacked attention tensor.
        tokens: List of token strings.
        layer: The layer index to visualize (0-indexed).
        head: The attention head index to visualize (0-indexed).
    """
    # Extract the specific attention matrix
    # Shape is (batch_size=1, num_heads, seq_len, seq_len). We take [0, head]
    attn_matrix = attention[layer, 0, head].numpy()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        attn_matrix, 
        xticklabels=tokens, 
        yticklabels=tokens, 
        cmap="viridis", 
        square=True
    )
    plt.title(f"Attention Weights (Layer {layer}, Head {head})")
    plt.xlabel("Key (Token being attended to)")
    plt.ylabel("Query (Token attending)")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    # Save the plot so it can be verified in a headless environment if needed
    plt.savefig(f"attention_l{layer}_h{head}.png")
    plt.show()

# --- Execution --- #
if __name__ == "__main__":
    # 1. Load Model
    model, tokenizer = load_model_and_tokenizer("gpt2")
    
    # 2. Define Input
    # A classic example demonstrating pronoun resolution
    sample_text = "The chef cooked the meal because he was hungry."
    print(f"\nProcessing text: '{sample_text}'")
    
    # 3. Get Attentions
    attentions, tokens = get_attention_weights(sample_text, model, tokenizer)
    print(f"Tokens: {tokens}")
    print(f"Attention tensor shape: {attentions.shape}")
    
    # 4. Visualize (Layer 0, Head 0 is often local/positional attention)
    plot_attention_head(attentions, tokens, layer=0, head=0)


## Common Pitfalls in Production

1.  **Memory Consumption:** In production inference, you almost *never* set `output_attentions=True`. The attention matrix is size $O(N^2)$ where $N$ is the sequence length. Storing this for a 4096-token context across 96 layers and 96 heads (like in larger models) will immediately cause Out-Of-Memory (OOM) errors.
2.  **Misinterpreting Weights:** A high attention weight doesn't always mean "this token is logically important." Heads specialize. Some heads just look at the previous token (positional), some look at punctuation, and some are "dead" heads that do nothing.
3.  **`torch.no_grad()`:** Always use `with torch.no_grad():` or `model.eval()` when doing inference/analysis. Forgetting this means PyTorch builds a computation graph for backpropagation, which wastes massive amounts of memory.

## Practical Lab / Homework

**Your Task:** 
We saw how to plot the attention weights. Now, write a function that programmatically finds the token that the word "he" attends to the most in a specific layer and head.

**Requirements:**
1. Create a function `find_max_attention(attention, tokens, query_token, layer, head)`.
2. It should find the index of the `query_token` in the `tokens` list.
3. It should extract the attention row for that token from the provided `layer` and `head`.
4. It should return the string token that has the highest attention weight (excluding itself).
5. Test it using `query_token="he"` on Layer 5, Head 8 (a head often associated with coreference resolution in GPT-2).

In [ ]:
def find_max_attention(
    attention: torch.Tensor, 
    tokens: List[str], 
    query_token: str, 
    layer: int, 
    head: int
) -> str:
    """
    Finds the token that the query_token attends to the most (excluding itself).
    """
    try:
        # Find the index of our query token
        query_idx = tokens.index(query_token)
    except ValueError:
        raise ValueError(f"Token '{query_token}' not found in sequence.")
        
    # Extract the specific row of attention for our query token
    # Shape: (seq_len,)
    attn_row = attention[layer, 0, head, query_idx, :].clone()
    
    # Exclude self-attention by setting it to a negative number
    attn_row[query_idx] = -1.0
    
    # Find the index with the maximum weight
    max_idx = torch.argmax(attn_row).item()
    
    return tokens[max_idx]

if __name__ == "__main__":
    # Using the attentions and tokens from the previous cell
    print("\n--- Lab Execution ---")
    target_token = "he"
    target_layer = 5
    target_head = 8
    
    most_attended = find_max_attention(
        attention=attentions, 
        tokens=tokens, 
        query_token=target_token, 
        layer=target_layer, 
        head=target_head
    )
    print(f"In Layer {target_layer}, Head {target_head}, the token '{target_token}' attends most to: '{most_attended}'")
